# 03b · Style attack: evaluation

**Stage 4b.** We score TF‑IDF + LR and DistilBERT (3 seeds) on six versions of the same 600 test articles:

| Condition | What changes compared with the previous row |
|---|---|
| `original` | Title + first ~300 words of the article |
| `stripped` | Source markers removed (agency tags, publisher suffixes, `[VIDEO]` tags, image credits, URLs) |
| `same` | LLM paraphrase that **keeps** the article's tone |
| `flip1` · `flip2` · `flip3` | LLM rewrite into the **opposite** tone at mild, clear and extreme intensity: real → sensational, fake → neutral |

This splits any accuracy drop into three parts:
**source-marker effect** (original → stripped), **paraphrase effect** (stripped → same) and the
**style effect** (same → flip), which is the quantity of interest.

Before scoring, an **NLI filter** checks that each rewrite kept the original's content. Main results use only
articles whose four rewrites all pass; results on all articles are reported as a robustness check.

**Runtime:** T4 GPU, roughly 20–30 minutes (most of it is the NLI filter, which is cached on Drive after the first run).

### ▶ Keep this tab awake (tap play once)

Phone browsers suspend background tabs, which can freeze the notebook's progress display or drop the
connection while long cells run. The player below loops a **near-silent** 5‑second sound. Tap ▶ once
after starting *Run all*; it keeps playing in your browser while the other cells run.
It does **not** extend Colab's own runtime limits, so keep the phone charging and Colab open.

In [ ]:
import io, wave, base64
import numpy as np
from IPython.display import HTML, display

_sr = 8000
_noise = (np.random.default_rng(0).standard_normal(_sr * 5) * 20).astype(np.int16)   # ~-64 dBFS: inaudible
_buf = io.BytesIO()
with wave.open(_buf, "wb") as _w:
    _w.setnchannels(1); _w.setsampwidth(2); _w.setframerate(_sr); _w.writeframes(_noise.tobytes())
display(HTML(f'<audio controls loop autoplay src="data:audio/wav;base64,{base64.b64encode(_buf.getvalue()).decode()}"></audio>'
             '<div style="font-size:12px;color:gray">Keep-alive player: tap ▶ if it is not already playing.</div>'))

## 0 · Configuration

In [ ]:
SEEDS = [42, 43, 44]
MAX_LEN = 256                         # same as notebook 02
NLI_MODEL = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
NLI_MAX_LEN = 512
NLI_BATCH = 32
COVERAGE_MIN = 0.70        # share of original sentences the rewrite must entail
CONTRA_MAX = 0.10          # max share of rewrite sentences that contradict the original
N_BOOT = 2000
CONDITIONS = ["original", "stripped", "same", "flip1", "flip2", "flip3"]
REWRITE_CONDITIONS = ["same", "flip1", "flip2", "flip3"]
N_CHUNKS = 24
PROJECT_DIRNAME = "NLP_style_attack"
REPO_RAW = "https://raw.githubusercontent.com/AfthabEK/NLP/HEAD"

## 1 · Setup

In [ ]:
import os, sys, json, re, time, urllib.request
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = f"/content/drive/MyDrive/{PROJECT_DIRNAME}"
else:
    BASE_DIR = os.environ.get("FAKESTYLE_BASE_DIR", os.path.abspath(f"./{PROJECT_DIRNAME}"))
DIRS = {k: os.path.join(BASE_DIR, k) for k in ["data", "results", "models", "figures"]}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

CODE_DIR = os.environ.get("FAKESTYLE_CODE_DIR") or "/content/fakestyle_src"
if not os.environ.get("FAKESTYLE_CODE_DIR"):
    os.makedirs(os.path.join(CODE_DIR, "fakestyle"), exist_ok=True)
    open(os.path.join(CODE_DIR, "fakestyle", "__init__.py"), "w").close()
    urllib.request.urlretrieve(f"{REPO_RAW}/src/fakestyle/text.py", os.path.join(CODE_DIR, "fakestyle", "text.py"))
sys.path.insert(0, CODE_DIR)
from fakestyle.text import strip_source_markers

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU (switch to a T4 runtime!)")
print("Outputs ->", BASE_DIR)

## 2 · Load the sample and the rewrites

In [ ]:
sample = pd.read_parquet(os.path.join(DIRS["data"], "attack_sample.parquet"))
# Recompute 'stripped' with the current version of the shared code (it gained a small fix after notebook 03a ran).
restripped = [strip_source_markers(ti, te) for ti, te in zip(sample["title_orig"], sample["text_orig"])]
sample["title_stripped"] = [r[0] for r in restripped]
sample["text_stripped"] = [r[1] for r in restripped]

local_dir = os.environ.get("FAKESTYLE_REWRITES_DIR")
rows, missing_chunks = [], []
for i in range(N_CHUNKS):
    name = f"chunk_{i:02d}.jsonl"
    try:
        if local_dir:
            raw = open(os.path.join(local_dir, name), encoding="utf-8").read()
        else:
            raw = urllib.request.urlopen(f"{REPO_RAW}/rewrites/{name}").read().decode("utf-8")
    except Exception:
        missing_chunks.append(name)
        continue
    rows += [json.loads(line) for line in raw.splitlines() if line.strip()]
rewrites = pd.DataFrame(rows).drop_duplicates(["id", "condition"], keep="last")
print(f"Loaded {len(rewrites):,} rewrites for {rewrites['id'].nunique()} articles; missing chunk files: {missing_chunks or 'none'}")

# One long table: one row per (article, condition)
versions = [sample[["id", "fake"]].assign(condition="original", title=sample["title_orig"], text=sample["text_orig"]),
            sample[["id", "fake"]].assign(condition="stripped", title=sample["title_stripped"], text=sample["text_stripped"]),
            rewrites.merge(sample[["id", "fake"]], on="id")[["id", "fake", "condition", "title", "text"]]]
long = pd.concat(versions, ignore_index=True)
complete_ids = set(rewrites.groupby("id")["condition"].nunique().loc[lambda s: s == len(REWRITE_CONDITIONS)].index)
print(f"Articles with all {len(REWRITE_CONDITIONS)} rewrites: {len(complete_ids)} of {len(sample)}")
long["words"] = long["text"].str.split().str.len()
print("\nMedian words per condition:")
print(long.groupby("condition")["words"].median().reindex(CONDITIONS).to_string())

### 2.1 · One example article in every condition

In [ ]:
ex_id = sorted(complete_ids)[0] if complete_ids else long["id"].iloc[0]
for c in CONDITIONS:
    r = long[(long["id"] == ex_id) & (long["condition"] == c)]
    if len(r):
        r = r.iloc[0]
        print(f"--- {c} ({'FAKE' if r['fake'] else 'REAL'}) ---\n{r['title']}\n{r['text'][:400]}...\n")

## 3 · NLI content check

Each article and its rewrite are split into sentences. With an NLI model we measure, for every rewrite:

- **coverage**: the share of *original* sentences entailed by the rewrite (was content kept?)
- **contradiction rate**: the share of *rewrite* sentences that the original contradicts (was content changed?)

A rewrite **passes** if coverage ≥ `COVERAGE_MIN` and contradiction rate ≤ `CONTRA_MAX`. Sentences that add
pure emotion ("This is outrageous!") count as *neutral*, not contradiction, so they do not fail a rewrite.
Scores are cached on Drive, so a re-run skips this step.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SENT_SPLIT = re.compile(r"(?:(?<=[.!?])|(?<=[.!?][\"”’']))\s+(?=[\"“‘'A-Z0-9])")
def sentences(text):
    return [s for s in (x.strip() for x in SENT_SPLIT.split(text)) if len(s.split()) >= 4]

nli_cache = os.path.join(DIRS["results"], "03b_nli_scores.parquet")
pairs = long[long["condition"].isin(REWRITE_CONDITIONS)].merge(
    sample[["id", "title_stripped", "text_stripped"]], on="id")

if os.path.exists(nli_cache) and set(map(tuple, pd.read_parquet(nli_cache)[["id", "condition"]].values)) >= set(map(tuple, pairs[["id", "condition"]].values)):
    nli = pd.read_parquet(nli_cache)
    print("Loaded cached NLI scores:", len(nli))
else:
    nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
    nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE).eval()
    label_idx = {v.lower(): int(k) for k, v in nli_model.config.id2label.items()}
    ENT, CON = label_idx["entailment"], label_idx["contradiction"]

    @torch.no_grad()
    def nli_probs(premises, hypotheses):
        out = []
        for s in range(0, len(premises), NLI_BATCH):
            enc = nli_tok(premises[s:s + NLI_BATCH], hypotheses[s:s + NLI_BATCH], truncation="only_first",
                          max_length=NLI_MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = nli_model(**enc).logits
            out.append(torch.softmax(logits.float(), -1).cpu().numpy())
        return np.concatenate(out) if out else np.zeros((0, 3))

    t0 = time.time()
    prem, hyp, owner = [], [], []          # owner = (row index, direction)
    for i, r in enumerate(pairs.itertuples()):
        for s in sentences(r.text_stripped):          # coverage: rewrite => original sentence
            prem.append(r.text); hyp.append(s); owner.append((i, "cov"))
        for s in sentences(r.text):                   # contradiction: original => rewrite sentence
            prem.append(r.text_stripped); hyp.append(s); owner.append((i, "con"))
    print(f"{len(prem):,} NLI pairs")
    probs = nli_probs(prem, hyp)
    per = {}
    for (i, d), p in zip(owner, probs):
        per.setdefault((i, d), []).append(p)
    recs = []
    for i, r in enumerate(pairs.itertuples()):
        cov = np.array(per.get((i, "cov"), [[1, 0, 0]]))
        con = np.array(per.get((i, "con"), [[1, 0, 0]]))
        recs.append({"id": r.id, "condition": r.condition,
                     "coverage": float((cov[:, ENT] > 0.5).mean()),
                     "contradiction_rate": float((con[:, CON] > 0.5).mean())})
    nli = pd.DataFrame(recs)
    nli.to_parquet(nli_cache)
    print(f"NLI done in {(time.time() - t0) / 60:.1f} min")
    del nli_model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

nli["passes"] = (nli["coverage"] >= COVERAGE_MIN) & (nli["contradiction_rate"] <= CONTRA_MAX)
nli = nli.merge(sample[["id", "fake"]], on="id")
nli_table = nli.groupby(["condition", "fake"]).agg(coverage=("coverage", "mean"),
                                                   contradiction_rate=("contradiction_rate", "mean"),
                                                   pass_rate=("passes", "mean")).round(3)
print(nli_table)
pass_ids = set(nli.groupby("id")["passes"].all().loc[lambda s: s].index) & complete_ids
print(f"\nArticles whose 4 rewrites all pass: {len(pass_ids)} "
      f"(real {sample[sample.id.isin(pass_ids)].fake.eq(0).sum()}, fake {sample[sample.id.isin(pass_ids)].fake.eq(1).sum()})")

MIN_MAIN = 100
MAIN_IDS = pass_ids if len(pass_ids) >= MIN_MAIN else complete_ids
if MAIN_IDS is not pass_ids:
    print(f"WARNING: fewer than {MIN_MAIN} articles pass the NLI check, so the main analysis uses all "
          f"{len(complete_ids)} complete articles. Inspect the NLI table above before trusting the results.")

## 4 · Score every version with both models

In [ ]:
import joblib
tfidf = joblib.load(os.path.join(DIRS["models"], "tfidf_lr.joblib"))
long["p_tfidf"] = tfidf["clf"].predict_proba(tfidf["vectorizer"].transform((long["title"] + "\n\n" + long["text"]).tolist()))[:, 1]

@torch.no_grad()
def bert_proba(model, tok, titles, texts, bs=64):
    model.eval()
    out = []
    for s in range(0, len(titles), bs):
        enc = tok(titles[s:s + bs], texts[s:s + bs], truncation="longest_first", max_length=MAX_LEN,
                  padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(**enc).logits
        out.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out)

for seed in SEEDS:
    mdir = os.path.join(DIRS["models"], f"distilbert_seed{seed}")
    tok = AutoTokenizer.from_pretrained(mdir)
    model = AutoModelForSequenceClassification.from_pretrained(mdir).to(DEVICE)
    long[f"p_bert{seed}"] = bert_proba(model, tok, long["title"].tolist(), long["text"].tolist())
    del model
    print(f"DistilBERT seed {seed} scored")
long["p_bert_mean"] = long[[f"p_bert{s}" for s in SEEDS]].mean(axis=1)
long.to_parquet(os.path.join(DIRS["results"], "03b_predictions.parquet"))

## 5 · Accuracy by condition

Main analysis: the articles whose four rewrites all passed the NLI check. DistilBERT is shown per seed and as the
mean of the three seeds' accuracies. 95% confidence intervals come from a bootstrap over articles.

In [ ]:
MODELS = {"TF-IDF + LR": ["p_tfidf"], "DistilBERT": [f"p_bert{s}" for s in SEEDS]}
rng = np.random.default_rng(0)

def correct_matrix(df, cols):
    # rows = articles (sorted by id), columns = model runs; 1 if the prediction is right
    d = df.sort_values("id")
    return np.stack([((d[c].values >= 0.5).astype(int) == d["fake"].values) for c in cols], axis=1).astype(float)

def acc_with_ci(df, cols):
    m = correct_matrix(df, cols).mean(axis=1)          # per-article accuracy averaged over runs
    boots = [m[rng.integers(0, len(m), len(m))].mean() for _ in range(N_BOOT)]
    return m.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)

def accuracy_table(ids):
    recs = []
    for model, cols in MODELS.items():
        for c in CONDITIONS:
            sub = long[(long["condition"] == c) & long["id"].isin(ids)]
            for cls, cls_name in [(None, "all"), (0, "real"), (1, "fake")]:
                part = sub if cls is None else sub[sub["fake"] == cls]
                if len(part) == 0:
                    continue
                a, lo, hi = acc_with_ci(part, cols)
                recs.append({"model": model, "condition": c, "class": cls_name, "accuracy": a,
                             "ci_low": lo, "ci_high": hi, "n": len(part)})
    return pd.DataFrame(recs)

acc_main = accuracy_table(MAIN_IDS)
acc_all = accuracy_table(complete_ids)
pivot = acc_main.pivot_table(index=["model", "class"], columns="condition", values="accuracy")[CONDITIONS]
print(f"Accuracy (%) on {len(MAIN_IDS)} articles ({'passing the NLI check' if MAIN_IDS is pass_ids else 'all complete articles'}):")
print((100 * pivot).round(1))

### 5.1 · Decomposition of the drop

In [ ]:
def decomposition(acc):
    piv = acc.pivot_table(index=["model", "class"], columns="condition", values="accuracy")
    out = pd.DataFrame({
        "source-marker effect (original→stripped)": piv["original"] - piv["stripped"],
        "paraphrase effect (stripped→same)": piv["stripped"] - piv["same"],
        **{f"style effect (same→{f})": piv["same"] - piv[f] for f in ["flip1", "flip2", "flip3"]},
        "total (original→flip3)": piv["original"] - piv["flip3"],
    })
    return (100 * out).round(1)

decomp = decomposition(acc_main)
print("Accuracy lost at each step, in percentage points (positive = the step hurt accuracy):")
decomp

### 5.2 · Flip rate and significance

**Flip rate**: among articles a model classifies correctly in `same`, the share it gets wrong in `flip3`.
**McNemar's exact test** compares `same` with each flip on the same articles (one test per model run).

In [ ]:
from scipy.stats import binomtest
sig = []
for model, cols in MODELS.items():
    for col in cols:
        base = long[(long["condition"] == "same") & long["id"].isin(MAIN_IDS)].sort_values("id")
        base_ok = (base[col].values >= 0.5).astype(int) == base["fake"].values
        for f in ["flip1", "flip2", "flip3"]:
            fl = long[(long["condition"] == f) & long["id"].isin(MAIN_IDS)].sort_values("id")
            fl_ok = (fl[col].values >= 0.5).astype(int) == fl["fake"].values
            b, c = int((base_ok & ~fl_ok).sum()), int((~base_ok & fl_ok).sum())
            sig.append({"model": model, "run": col, "flip": f,
                        "flip_rate": (base_ok & ~fl_ok).sum() / max(base_ok.sum(), 1),
                        "broken": b, "fixed": c,
                        "mcnemar_p": binomtest(b, b + c, 0.5).pvalue if b + c else 1.0})
sig_df = pd.DataFrame(sig)
print(sig_df.round(4).to_string(index=False))

## 6 · Dose–response: how far does P(fake) move?

In [ ]:
order = ["stripped", "same", "flip1", "flip2", "flip3"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (cls, cls_name, arrow) in zip(axes, [(0, "real articles → more sensational", "↑ = fooled"),
                                             (1, "fake articles → more neutral", "↓ = fooled")]):
    sub = long[long["id"].isin(MAIN_IDS) & (long["fake"] == cls)]
    for label, col, style in [("TF-IDF + LR", "p_tfidf", "o-"), ("DistilBERT (mean of seeds)", "p_bert_mean", "s-")]:
        means = sub.groupby("condition")[col].mean().reindex(order)
        ax.plot(order, means.values, style, label=label)
    ax.axhline(0.5, color="gray", lw=0.8, ls="--")
    ax.set_title(f"{cls_name}\n(mean P(fake); {arrow})"); ax.set_ylim(0, 1); ax.set_xlabel("condition")
axes[0].set_ylabel("mean predicted P(fake)"); axes[0].legend(loc="upper left")
fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], "03b_dose_response.png"), dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for model, style in [("TF-IDF + LR", "o-"), ("DistilBERT", "s-")]:
    a = acc_main[(acc_main["model"] == model) & (acc_main["class"] == "all")].set_index("condition").reindex(CONDITIONS)
    ax.plot(CONDITIONS, 100 * a["accuracy"], style, label=model)
    ax.fill_between(CONDITIONS, 100 * a["ci_low"], 100 * a["ci_high"], alpha=0.15)
ax.axhline(50, color="gray", lw=0.8, ls="--"); ax.set_ylabel("accuracy (%)"); ax.set_ylim(0, 100)
ax.set_title("Accuracy as the style is flipped (95% bootstrap CI)"); ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(DIRS["figures"], "03b_accuracy_by_condition.png"), dpi=150); plt.show()

## 7 · Robustness check: all articles, ignoring the NLI filter

In [ ]:
print(decomposition(acc_all))

## 8 · Examples the models got wrong after the flip

In [ ]:
wide = long[long["id"].isin(MAIN_IDS)].pivot_table(index=["id", "fake"], columns="condition", values="p_bert_mean").reset_index()
fooled = wide[((wide["fake"] == 0) & (wide["same"] < 0.5) & (wide["flip3"] >= 0.5)) |
              ((wide["fake"] == 1) & (wide["same"] >= 0.5) & (wide["flip3"] < 0.5))]
print(f"Articles where DistilBERT (mean) was right on 'same' but wrong on 'flip3': {len(fooled)}")
for aid in fooled["id"].head(3):
    for c in ["same", "flip3"]:
        r = long[(long["id"] == aid) & (long["condition"] == c)].iloc[0]
        print(f"[{aid}] {'FAKE' if r['fake'] else 'REAL'} · {c} · P(fake): tfidf={r['p_tfidf']:.2f} bert={r['p_bert_mean']:.2f}")
        print(f"   {r['title']}\n   {r['text'][:300]}...\n")

## 9 · Save summary

In [ ]:
summary = {
    "notebook": "03b_style_attack",
    "config": {"SEEDS": SEEDS, "MAX_LEN": MAX_LEN, "NLI_MODEL": NLI_MODEL, "COVERAGE_MIN": COVERAGE_MIN,
               "CONTRA_MAX": CONTRA_MAX, "N_BOOT": N_BOOT},
    "n_rewrites": int(len(rewrites)), "missing_chunks": missing_chunks,
    "n_complete": len(complete_ids), "n_pass_nli": len(pass_ids), "main_uses_nli_filter": MAIN_IDS is pass_ids,
    "nli_by_condition": nli_table.reset_index().to_dict(orient="records"),
    "accuracy_main": acc_main.round(4).to_dict(orient="records"),
    "decomposition_main_pp": decomp.reset_index().to_dict(orient="records"),
    "decomposition_all_pp": decomposition(acc_all).reset_index().to_dict(orient="records"),
    "significance": sig_df.round(6).to_dict(orient="records"),
    "n_fooled_bert_mean_flip3": int(len(fooled)),
}
with open(os.path.join(DIRS["results"], "03b_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=float)
print("Saved results/03b_summary.json")

**Done.** New in `MyDrive/NLP_style_attack/`: `results/03b_summary.json`, `results/03b_predictions.parquet`
(every article × condition × model), `results/03b_nli_scores.parquet`, and two figures in `figures/`.